In [ ]:
import pandas as pd
import nltk
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import rotate_axes

nltk.download('punkt_tab') #for summaC


In [ ]:
from summac.model_summac import SummaCZS, SummaCConv

model_zs = SummaCZS(granularity="sentence", model_name="vitc", device="cpu") # If you have a GPU: switch to: device="cuda"
model_conv = SummaCConv(models=["vitc"], bins='percentile', granularity="sentence", nli_labels="e", device="cpu", start_file="default", agg="mean")

document = """Scientists are studying Mars to learn about the Red Planet and find landing sites for future missions.
One possible site, known as Arcadia Planitia, is covered instrange sinuous features.
The shapes could be signs that the area is actually made of glaciers, which are large masses of slow-moving ice.
Arcadia Planitia is in Mars' northern lowlands."""

summary1 = "There are strange shape patterns on Arcadia Planitia. The shapes could indicate the area might be made of glaciers. This makes Arcadia Planitia ideal for future missions."
score_zs1 = model_zs.score([document], [summary1])
score_conv1 = model_conv.score([document], [summary1])
print("[Summary 1] SummaCZS Score: %.3f; SummacConv score: %.3f" % (score_zs1["scores"][0], score_conv1["scores"][0])) # [Summary 1] SummaCZS Score: 0.582; SummacConv score: 0.536

summary2 = "There are strange shape patterns on Arcadia Planitia. The shapes could indicate the area might be made of glaciers."
score_zs2 = model_zs.score([document], [summary2])
score_conv2 = model_conv.score([document], [summary2])
print("[Summary 2] SummaCZS Score: %.3f; SummacConv score: %.3f" % (score_zs2["scores"][0], score_conv2["scores"][0])) # [Summary 2] SummaCZS Score: 0.877; SummacConv score: 0.709

#source: https://github.com/tingofurro/summac/blob/master/README.md

In [ ]:
from summac.model_summac import SummaCZS

model_CZS = SummaCZS(granularity="sentence", model_name="vitc")
model_CONV = SummaCConv(models=["vitc"], bins='percentile', granularity="sentence", nli_labels="e", device="cpu", start_file="default", agg="mean")


original = "The Distributor shall pay the Company a fee of $500 per month."
faithful = "The Distributor pays the Company $500 every month."
corrupted = "The Distributor may pay the Company $500 every month."  # shall -> may

print("CZS: ")
print(model_CZS.score([original], [faithful]))
print(model_CZS.score([original], [corrupted]))
print("CONV: ")
print(model_CONV.score([original], [faithful]))
print(model_CONV.score([original], [corrupted]))

#testing how SummaC reacts to modal changes

In [ ]:
#Generated examples with Claude Haiku -> checked by me

test_pairs_modal_flip= [
    # modal_flip
    {"original": "The Contractor shall complete the work within 30 days.",
     "faithful": "The Contractor is required to finish the work within 30 days.",
     "corrupted": "The Contractor may complete the work within 30 days.",
     "error_type": "modal_flip"},
    {"original": "The Licensee must obtain written consent before sublicensing.",
     "faithful": "The Licensee has to get written permission before sublicensing.",
     "corrupted": "The Licensee should obtain written consent before sublicensing.",
     "error_type": "modal_flip"},
    {"original": "The Buyer shall inspect the goods within five business days.",
     "faithful": "The Buyer is obligated to inspect the goods within five business days.",
     "corrupted": "The Buyer can inspect the goods within five business days.",
     "error_type": "modal_flip"},
    {"original": "The Company will indemnify the Partner against all third-party claims.",
     "faithful": "The Company agrees to indemnify the Partner for all third-party claims.",
     "corrupted": "The Company may indemnify the Partner against all third-party claims.",
     "error_type": "modal_flip"}
]
test_pairs_dropped_clause = [
    # dropped_clause
    {"original": "The Landlord may enter the premises for inspection, provided that 24 hours' notice is given.",
     "faithful": "The Landlord can enter for inspection if they give 24 hours' notice first.",
     "corrupted": "The Landlord may enter the premises for inspection.",
     "error_type": "dropped_clause"},
    {"original": "Either party may terminate this Agreement upon 60 days' written notice, unless a material breach has occurred.",
     "faithful": "Either party can end the agreement with 60 days' written notice, except in cases of a serious breach.",
     "corrupted": "Either party may terminate this Agreement upon 60 days' written notice.",
     "error_type": "dropped_clause"},
    {"original": "The Vendor shall not be liable for delays caused by events beyond its reasonable control, including natural disasters.",
     "faithful": "The Vendor isn't responsible for delays from things outside its control, like natural disasters.",
     "corrupted": "The Vendor shall not be liable for delays.",
     "error_type": "dropped_clause"},
    {"original": "Payment is due within 30 days, except where the Buyer disputes the invoice in good faith.",
     "faithful": "Payment must be made within 30 days, unless the Buyer has a genuine dispute about the invoice.",
     "corrupted": "Payment is due within 30 days.",
     "error_type": "dropped_clause"}
    ]
test_pairs_added_content = [
    # added_content
    {"original": "The Distributor shall pay the Company a fee of $500 per month.",
     "faithful": "The Distributor pays the Company $500 every month.",
     "corrupted": "The Distributor shall pay the Company a fee of $500 per month, plus a 5% late penalty.",
     "error_type": "added_content"},
    {"original": "This Agreement is governed by the laws of the State of Delaware.",
     "faithful": "Delaware law governs this Agreement.",
     "corrupted": "This Agreement is governed by the laws of the State of Delaware and must be arbitrated in New York.",
     "error_type": "added_content"},
    {"original": "The Employee shall receive fifteen days of paid vacation per year.",
     "faithful": "The Employee gets 15 paid vacation days annually.",
     "corrupted": "The Employee shall receive fifteen days of paid vacation per year, which can be carried over indefinitely.",
     "error_type": "added_content"},
    {"original": "The Supplier warrants that the goods are free from material defects.",
     "faithful": "The Supplier guarantees the goods have no major defects.",
     "corrupted": "The Supplier warrants that the goods are free from material defects for a period of five years.",
     "error_type": "added_content"}]
test_pairs_negation = [
    # negation
    {"original": "The Parties agree that this Agreement is not assignable without prior written consent.",
     "faithful": "Neither party can transfer this Agreement without getting written permission first.",
     "corrupted": "The Parties agree that this Agreement is assignable without prior written consent.",
     "error_type": "negation"},
    {"original": "The Licensor does not grant any rights to use the Trademark.",
     "faithful": "The Licensor doesn't give permission to use the Trademark.",
     "corrupted": "The Licensor grants rights to use the Trademark.",
     "error_type": "negation"},
    {"original": "No warranty is made regarding the accuracy of the information provided.",
     "faithful": "There's no guarantee that the information provided is accurate.",
     "corrupted": "A warranty is made regarding the accuracy of the information provided.",
     "error_type": "negation"},
    {"original": "The Tenant shall not sublet the premises without the Landlord's approval.",
     "faithful": "The Tenant can't sublet the property without the Landlord's okay.",
     "corrupted": "The Tenant shall sublet the premises without the Landlord's approval.",
     "error_type": "negation"},
]

for p in test_pairs_modal_flip:
    print("modal flip")
    print("CZS: ")
    print(model_CZS.score([p["original"]], [p["faithful"]])["scores"])
    print(model_CZS.score([p["original"]], [p["corrupted"]])["scores"])
    print("CONV: ")
    print(model_CONV.score([p["original"]], [p["faithful"]])["scores"])
    print(model_CONV.score([p["original"]], [p["corrupted"]])["scores"])
for p in test_pairs_dropped_clause:
    print("dropped clause")
    print("CZS: ")
    print(model_CZS.score([p["original"]], [p["faithful"]])["scores"])
    print(model_CZS.score([p["original"]], [p["corrupted"]])["scores"])
    print("CONV: ")
    print(model_CONV.score([p["original"]], [p["faithful"]])["scores"])
    print(model_CONV.score([p["original"]], [p["corrupted"]])["scores"])
for p in test_pairs_added_content:
    print("added content")
    print("CZS: ")
    print(model_CZS.score([p["original"]], [p["faithful"]])["scores"])
    print(model_CZS.score([p["original"]], [p["corrupted"]])["scores"])
    print("CONV: ")
    print(model_CONV.score([p["original"]], [p["faithful"]])["scores"])
    print(model_CONV.score([p["original"]], [p["corrupted"]])["scores"])
for p in test_pairs_negation:
    print("negation")
    print("CZS: ")
    print(model_CZS.score([p["original"]], [p["faithful"]])["scores"])
    print(model_CZS.score([p["original"]], [p["corrupted"]])["scores"])
    print("CONV: ")
    print(model_CONV.score([p["original"]], [p["faithful"]])["scores"])
    print(model_CONV.score([p["original"]], [p["corrupted"]])["scores"])

    #checking how SummaC performs on the examples

In [ ]:
records1 = []
test_pairs = [test_pairs_negation, test_pairs_added_content, test_pairs_dropped_clause, test_pairs_modal_flip]
test_pairs_names = ["negation", "added content", "dropped clause", "modal flip"]

for i in range(len(test_pairs)):
    for p in test_pairs[i]:
        czs_f = model_CZS.score([p["original"]], [p["faithful"]])["scores"][0]
        czs_c = model_CZS.score([p["original"]], [p["corrupted"]])["scores"][0]
        conv_f = model_CONV.score([p["original"]], [p["faithful"]])["scores"][0]
        conv_c = model_CONV.score([p["original"]], [p["corrupted"]])["scores"][0]
        records1.append({"category": test_pairs_names[i], "model": "CZS", "faithful": czs_f, "corrupted": czs_c})
        records1.append({"category": test_pairs_names[i], "model": "CONV", "faithful": conv_f, "corrupted": conv_c})

scores_df1 = pd.DataFrame(records1)
scores_df1.head(50)

In [ ]:
scores_df1["difference"] = scores_df1["faithful"] - scores_df1["corrupted"]
summed_aggregate = scores_df1.groupby(["category", "model"])["difference"].mean().unstack()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
summed_aggregate["CZS"].plot(kind="bar", ax=axes[0], title="CZS")
summed_aggregate["CONV"].plot(kind="bar", ax=axes[1], title="CONV")
plt.tight_layout()
plt.savefig("../results/faithfulness_filter_reverse_test.pdf")
plt.show()

Here can be seen, that for the CZS the SummaC tool can not detect change in the modals and in the case of the CONV model even shows a negative pattern. The hypothesis is that because SummaC always compares the summary to the leagal text and not vice versa. So the as long as the summary is covered by the original it will be give a good score.

In [ ]:
#same process, but here also running SummaC with the summary as the Original

records2 = []
test_pairs = [test_pairs_negation, test_pairs_added_content, test_pairs_dropped_clause, test_pairs_modal_flip]
test_pairs_names = ["negation", "added content", "dropped clause", "modal flip"]

for i in range(len(test_pairs)):
    for p in test_pairs[i]:
        czs_f = model_CZS.score([p["original"]], [p["faithful"]])["scores"][0]
        czs_c = model_CZS.score([p["original"]], [p["corrupted"]])["scores"][0]
        czs_f_rev = model_CZS.score([p["faithful"]], [p["original"]])["scores"][0]
        czs_c_rev = model_CZS.score([p["corrupted"]], [p["original"]])["scores"][0]
        conv_f = model_CONV.score([p["original"]], [p["faithful"]])["scores"][0]
        conv_c = model_CONV.score([p["original"]], [p["corrupted"]])["scores"][0]
        conv_f_rev = model_CONV.score([p["faithful"]], [p["original"]])["scores"][0]
        conv_c_rev = model_CONV.score([p["corrupted"]], [p["original"]])["scores"][0]
        records2.append({"category": test_pairs_names[i], "model": "CZS", "faithful": czs_f, "corrupted": czs_c})
        records2.append({"category": test_pairs_names[i], "model": "CONV_reverse", "faithful": conv_f_rev, "corrupted": conv_c_rev})
        records2.append({"category": test_pairs_names[i], "model": "CZS_reverse", "faithful": czs_f_rev, "corrupted": czs_c_rev})
        records2.append({"category": test_pairs_names[i], "model": "CONV", "faithful": conv_f, "corrupted": conv_c})

scores_df = pd.DataFrame(records2)
scores_df.head(50)

In [ ]:
scores_df["gap"] = scores_df["faithful"] - scores_df["corrupted"]
summed_aggregate = scores_df.groupby(["category", "model"])["gap"].mean().unstack()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
summed_aggregate[["CZS", "CZS_reverse"]].plot(kind="bar", ax=axes[0], title="CZS: forward vs reverse")
summed_aggregate[["CONV", "CONV_reverse"]].plot(kind="bar", ax=axes[1], title="CONV: forward vs reverse")
plt.tight_layout()
plt.savefig("../results/faithfulness_filter_reverse_test.pdf")
plt.show()
#in the plots you can see, that the reverse SummaC catches the dropped content much better. Added content has the oposite pattern - as expected.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(6, 8))

summed_aggregate["CZS"].plot(kind="bar", ax=axes[0], color="blue")
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_ylabel("faithful score - corrupted score")
axes[0].set_title("CZS: separation by category")

summed_aggregate["CONV"].plot(kind="bar", ax=axes[1], color="orange")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("CONV: separation by category")

plt.tight_layout()
plt.savefig("../results/separation_comparison.pdf")
plt.show()
#I will be using CZS as it is clearly better at differentiating

In [ ]:
czs_scores = scores_df[scores_df["model"].isin(["CZS", "CZS_reverse"])]
#I am now selecting a threshold for my filter
#as modal flips can not be reliably caught I will not include them as a criteria
added = czs_scores[scores_df["category"] == "added content"]
dropped = czs_scores[scores_df["category"] == "dropped clause"]
negation = czs_scores[scores_df["category"] == "negation"]


In [ ]:


#added content
summary_added = added.groupby("model").agg(faithful_min=("faithful", "min"), corrupted_max=("corrupted", "max"))
summary_added["category"] = "added content"

#dropped clause
summary_dropped = dropped.groupby("model").agg(faithful_min=("faithful", "min"), corrupted_max=("corrupted", "max"))
summary_dropped["category"] = "dropped clause"

#negation
summary_negation = negation.groupby("model").agg(faithful_min=("faithful", "min"), corrupted_max=("corrupted", "max"))
summary_negation["category"] = "negation"


In [ ]:
summary = pd.concat([summary_added, summary_dropped, summary_negation])
summary.to_csv("../results/summac_summary_per_category.csv")
print(summary)


In [ ]:
import textstat
from src.readability_filter import readability_drop
#using the readabily_filter file

In [ ]:
tos_train = pd.read_csv("../data/tos_train.csv")
print(len(tos_train))
#getting the tos:dr train data

In [ ]:
grade_drops = []
for i, row in tos_train.iterrows():
    drop = readability_drop(row["original_text"], row["reference_summary"])
    grade_drops.append(drop)

tos_train["grade_drop"] = grade_drops

In [ ]:
tos_train.boxplot(column="grade_drop")
plt.ylabel("grade level drop")
plt.title("Readability drop: original vs. reference summary")
plt.savefig("../results/textstat_tldr_data.pdf")
plt.show()

In [ ]:

print(tos_train["grade_drop"].describe())
tos_train["grade_drop"].sort_values().head(354)

#25% = 2.104911 as threshold for readability